# Estruturação dos registros

Este notebook junta os textos limpos do notebook 02 em `data/processed/precedentes.jsonl`. Cada precedente vira um registro com os metadados usados nos filtros da busca, como citação, situação, vigência e link.

In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
INTERIM = ROOT / "data/interim"
PROCESSED = ROOT / "data/processed"
PROCESSED.mkdir(parents=True, exist_ok=True)


def ler_jsonl(origem: Path) -> list[dict]:
    return [json.loads(linha) for linha in origem.read_text(encoding="utf-8").splitlines()]

## Súmulas

In [2]:
SITUACOES = {
    None: "vigente",
    "SÚMULA ALTERADA": "alterada",
    "SÚMULA CANCELADA": "cancelada",
    "SÚMULA REVOGADA": "revogada",
}

sumulas = []
for s in ler_jsonl(INTERIM / "sumulas_limpas.jsonl"):
    numero, situacao = s["numero"], SITUACOES[s["marca"]]
    sumulas.append(
        {
            "id": f"stj:sumula:{numero}",
            "citacao": f"Súmula {numero}/STJ",
            "fonte": "sumula",
            "tipo": "Súmula",
            "numero": numero,
            "enunciado": s["enunciado"],
            "questao": "",
            "situacao": situacao,
            "vigente": situacao in ("vigente", "alterada"),
            "tem_tese": True,
            "referencia": s["citacao"],
            "historico": s["historico"],
            "assuntos": "",
            "url": f"https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula={numero}",
        }
    )
print("Súmulas:", len(sumulas), "|", Counter(s["situacao"] for s in sumulas))

Súmulas: 676 | Counter({'vigente': 641, 'cancelada': 29, 'alterada': 5, 'revogada': 1})


## Precedentes qualificados

`vigente=False` marca os precedentes que deixaram de valer por si, seja porque foram cancelados, prejudicados ou absorvidos por outro tema. `tem_tese=False` marca os que ainda estão em andamento. Neles, só a questão foi afetada e ainda não há entendimento firmado, então podem aparecer como afins, mas não como consonância.

In [3]:
SITUACOES_INATIVAS = {
    "cancelada", "cancelado", "vinculada a tema", "vinculada a tema repetitivo", "prejudicada",
    "sem processo vinculado", "suspensão indeferida", "finalizada determinação de suspensão",
}  # fmt: skip

precedentes = []
for t in ler_jsonl(INTERIM / "temas_limpos.jsonl"):
    tipo = t["tipo"] or "Relevância"
    numero = t["numero"]
    precedentes.append(
        {
            "id": f"stj:{tipo.lower()}:{numero}",
            "citacao": f"{tipo} {numero}/STJ",
            "fonte": "precedente",
            "tipo": tipo,
            "numero": numero,
            "enunciado": t["tese"] or t["questao"],
            "questao": t["questao"],
            "situacao": t["situacao"],
            "vigente": t["situacao"].lower() not in SITUACOES_INATIVAS,
            "tem_tese": bool(t["tese"]),
            "referencia": t["orgao"],
            "historico": t["entendimento_anterior"],
            "assuntos": t["assuntos"],
            "url": (
                "https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp"
                f"?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial={numero}&cod_tema_final={numero}"
                if tipo == "Tema"
                else ""
            ),
        }
    )

assert len({p["id"] for p in precedentes}) == len(precedentes), "id de precedente repetido"
print("Precedentes:", len(precedentes))
print(pd.DataFrame(precedentes).groupby(["tipo", "vigente", "tem_tese"]).size().to_string())

Precedentes: 2369
tipo          vigente  tem_tese
Controvérsia  False    False        803
                       True           2
              True     False         25
IAC           False    False          1
              True     False          4
                       True          17
PUIL          False    False          1
              True     False          7
                       True          19
Relevância    True     False          3
                       True           2
SIRDR         False    False         10
              True     False          1
Tema          False    False        194
                       True           4
              True     False        129
                       True        1147


## Gravação

In [4]:
registros = sumulas + precedentes
saida = PROCESSED / "precedentes.jsonl"
with saida.open("w", encoding="utf-8") as arquivo:
    for registro in registros:
        arquivo.write(json.dumps(registro, ensure_ascii=False) + "\n")
print(f"{saida.relative_to(ROOT)}: {len(registros)} registros")

data/processed/precedentes.jsonl: 3045 registros
